# Machine learning

Nesta etapa, será desenvolvido um modelo de Machine Learning utilizando os dados do arquivo sinistros_sp_2025_limpo_eda. O objetivo é treinar um modelo de classificação capaz de identificar os acidentes de trânsito como fatais ou não fatais, buscando analisar os fatores associados à gravidade dos sinistros no município de São Paulo em 2025.

Ao longo do notebook, serão realizadas etapas de preparação dos dados, seleção das métricas e da variável-alvo, divisão dos dados em conjuntos de treino e teste, tratamento do desbalanceamento das classes, treinamento dos modelos e avaliação de seus resultados. Por fim, serão analisadas as métricas de desempenho para verificar a capacidade do modelo de classificar corretamente os sinistros.

## Importação das bibliotecas

In [1]:
# Manipulação de Dados
import pandas as pd
import numpy as np

# Visualização Gráfica
import matplotlib.pyplot as plt

#Separação dos dados: treino e teste
from sklearn.model_selection import train_test_split 

# Balanceamneto das classes
from imblearn.over_sampling import SMOTE

# Codificação de variáveis categóricas
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Modelos de Aprendizado de Máquina
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeRegressor

# Métricas de avaliação dos modelos
from sklearn.metrics import (  #Métricas de Avaliação
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    roc_curve,
    roc_auc_score,
    precision_recall_curve
) 

## Carregamento dos dados

In [3]:
# Carregar a base
df = pd.read_csv(
    "../data/processed/sinistros_sp_2025_limpo_eda.csv",
    sep=";",
    decimal=",",
    encoding="utf-8-sig",
    low_memory=False
)

# mostrar os 5 primeiros registros
df.head()

,id_sinistro,tipo_registro,data_sinistro,ano_sinistro,mes_sinistro,dia_sinistro,hora_sinistro,ano_mes_sinistro,dia_da_semana,turno,...,tp_sinistro_choque,tp_sinistro_atrop_animal,tp_sinistro_capotamento,tp_sinistro_engavetamento,tp_sinistro_tombamento,tp_sinistro_outros,tp_sinistro_nao_disponivel,hora,latitude_num,longitude_num
0,2523341,NOTIFICACAO,01/01/2025,2025,1,1,03:11,2025/01,Quarta-feira,MADRUGADA,...,NaN,NaN,NaN,NaN,NaN,NaN,S,NaN,NaN,NaN
1,2467785,SINISTRO FATAL,01/01/2025,2025,1,1,02:40,2025/01,Quarta-feira,MADRUGADA,...,S,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-23.775436541,-46.6817490556
2,2517646,NOTIFICACAO,01/01/2025,2025,1,1,01:15,2025/01,Quarta-feira,MADRUGADA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2524480,SINISTRO NAO FATAL,01/01/2025,2025,1,1,03:08,2025/01,Quarta-feira,MADRUGADA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-23.5078665,-46.4917972
4,2531534,SINISTRO NAO FATAL,02/01/2025,2025,1,2,20:40,2025/01,Quinta-feira,NOITE,...,NaN,NaN,NaN,NaN,NaN,S,NaN,NaN,-23.5755,-46.7129


## Definição da Variável-Alvo (fatal) e Remoção de Data Leakage

A variável-alvo do modelo será criada a partir da coluna `tipo_registro`. Inicialmente, serão mantidos apenas os registros classificados como **SINISTRO FATAL** e **SINISTRO NAO FATAL**, excluindo as notificações que não possuem uma classificação de gravidade definida. Em seguida, será criada a coluna **fatal**, que representará a gravidade do acidente de forma binária: 1 para sinistros **fatais** e 0 para sinistros **não fatais**. Por fim, será verificada a distribuição das classes para avaliar a quantidade de registros em cada categoria.

In [6]:
# Manter somente os registros classificados (fatal e não fatal)
df = df[
    df["tipo_registro"].isin([
        "SINISTRO FATAL",
        "SINISTRO NAO FATAL"
    ])
]

df["tipo_registro"].value_counts()

tipo_registro
SINISTRO NAO FATAL    22856
SINISTRO FATAL          882
Name: count, dtype: int64

In [9]:
# Criar a variável-alvo (fatal)
df["fatal"] = (
    df["tipo_registro"] == "SINISTRO FATAL"
).astype(int)

# Conferir a distribuição do alvo
print(df["fatal"].value_counts())

fatal
0    22856
1      882
Name: count, dtype: int64


### Tratamento de Data Leakage e remoção de colunas irrelevantes

Nesta etapa, serão removidas as colunas que possam comprometer a confiabilidade do modelo de classificação. Para evitar o data leakage (vazamento de dados), serão excluídas variáveis que contenham informações diretamente relacionadas à gravidade do acidente, pois podem revelar indiretamente a variável-alvo fatal. Também serão removidos identificadores únicos, informações constantes ou redundantes e colunas totalmente vazias. Além disso, serão descartadas as versões originais de latitude e longitude, ja que possuam suas versões numéricas disponíveis. O objetivo é reduzir informações desnecessárias e preparar o conjunto de dados para o treinamento do modelo.

In [ ]:
# Remover identificadores e variáveis com vazamento de informação (Data Leakage)
colunas_remover = [
    # Alvo original e vazamento de dados
    "tipo_registro",
    "qtd_gravidade_fatal",
    "qtd_gravidade_grave",
    "qtd_gravidade_leve",
    "qtd_gravidade_ileso",
    "qtd_gravidade_nao_disponivel",

    # Identificação e informações constantes
    "id_sinistro",
    "cod_ibge",
    "municipio",
    "ano_sinistro",
    "regiao_administrativa",

    # Colunas vazias
    "hora",
    "tp_sinistro_colisao_traseira",

    # Colunas com informações redundantes
    "latitude_num",
    "longitude_num",
    "ano_mes_sinistro",
    "data_sinistro"
]

# Remoção das colunas
df = df.drop(
    columns=colunas_remover,
    errors="ignore"  # Evita um erro caso algum nome da lista não exista no DataFrame.
)

print("Dimensão final:", df.shape)

### Tratamento das colunas com valores nulos

Os valores nulos das colunas `qtd_*`. De acordo com o dicionário de dados, esses valores não podem representar ausência de informação ou quantidade (0), em relação aos acidentes. Vale também para as colunas `longitude`, `latitude`, `hora_sinistro`, `logradouro` e `num_logradouro`.

Já na coluna `tipo_local` os valores nulos representam, de acordo com dicionário de dados, "NAO DISPONIVEL".

In [ ]:
# Substituir os valores nulos por "NAO DISPONIVEL"
df["tipo_local"] = df["tipo_local"].fillna("NAO DISPONIVEL")

# Mostrar os dados da coluna "tipo_local"
print(df["tipo_local"].value_counts())

## Separação dos Dados em Treino e Teste

O conjunto de dados será dividido em duas partes: **80% para treinamento e 20% para teste**. Essa separação permite que os modelos de Machine Learning aprendam padrões a partir dos dados de treino e, posteriormente, sejam avaliados utilizando dados que não foram utilizados durante o treinamento.

A metodologia escolhida será o método *Holdout*, utilizando a função **train_test_split** do Scikit-learn. O parâmetro **test_size=0.20** define a proporção destinada ao teste, enquanto **random_state=42** garante a reprodutibilidade da divisão. Além disso, será utilizado **stratify=y** para manter aproximadamente a mesma proporção de sinistros fatais e não fatais nos dois conjuntos, considerando o desbalanceamento entre as classes.

In [ ]:
# Separar as features do alvo
X = df.drop(columns=["fatal"])
y = df["fatal"]

# Dividir treino e teste
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Conferir a distribuição
print("Treino:")
print(y_train.value_counts(normalize=True))

print("\nTeste:")
print(y_test.value_counts(normalize=True))

## Pré-processamento

Nesta etapa, os dados serão preparados para o treinamento dos modelos, incluindo o tratamento de valores ausentes, a codificação de variáveis categóricas e a padronização das informações, quando necessário. O objetivo é garantir que os dados estejam adequados para os algoritmos de classificação.

### Binarização das variáveis de tipo de sinistro

Transformação das variáveis específicas `tp_sinistro_...` em valores binários, atribuindo 1 para indicar a **presença** do tipo de sinistro e 0 para indicar sua **ausência**, de acordo com o dicionário de dados. Essa conversão padroniza os dados categóricos para facilitar sua utilização no treinamento do modelo de Machine Learning.

In [1]:
# Lista todas as colunas relacionadas ao tipo de sinistro
colunas_sinistro = [
    col for col in df.columns
    if col.startswith("tp_sinistro_")
]

print(colunas_sinistro)

NameError: name 'df' is not defined

In [ ]:
# Confere os valores de cada coluna
for col in colunas_sinistro:
    print(f"\n{col}")
    print(df[col].value_counts(dropna=False).head(10))

In [1]:
# Separa a coluna "tp_sinistro_primario"
colunas_indicadoras = [
    col for col in df.columns
    if col.startswith("tp_sinistro_")
    and col != "tp_sinistro_primario"
]

# Transforma S em 1 e campos vazios em 0, conforme o dicionário.
for col in colunas_indicadoras:
    df[col] = (
        df[col]
        .fillna(0)
        .astype(int)
    )

# Mostrar os dados das colunas "tp_sinistro_*"
display(df[colunas_indicadoras].head())

NameError: name 'df' is not defined

### Feature Engineering (engenharia de atributos)

Consiste na criação, transformação ou seleção de atributos a partir dos dados existentes, buscando destacar informações relevantes para o modelo. Neste projeto, essa etapa poderá ajudar a representar melhor características dos sinistros e suas possíveis relações com a gravidade dos acidentes.

In [ ]:
# Converter a coluna "hora_sinistro" para formato de horário
hora = pd.to_datetime(
    df["hora_sinistro"],
    format="%H:%M",
    errors="coerce"
).dt.hour

# Criar a coluna "turno"
df["turno"] = pd.cut(
    hora,
    bins=[-1, 5, 11, 17, 23],
    labels=["Madrugada", "Manhã", "Tarde", "Noite"]
)

# Mostrar os dados da coluna "turno"
print(df["turno"].value_counts(dropna=False))

### Codificação de variáveis categóricas (One-Hot Encoding)

Nesta etapa, as variáveis categóricas serão convertidas em valores numéricos por meio de técnicas como o One-Hot Encoding, permitindo que os modelos de Machine Learning processem informações como turno, dia da semana e tipo de acidente.

In [ ]:
# Identificar as colunas categóricas
colunas_categoricas = X_train.select_dtypes(
    include=["object", "category", "bool"]
).columns

# Criar o codificador
preprocessador = ColumnTransformer(
    transformers=[
        (
            "categoricas",
            OneHotEncoder(handle_unknown="ignore"),
            colunas_categoricas
        )
    ],
    remainder="passthrough"
)

# Ajustar com os dados de treino e transformar
X_train_codificado = preprocessador.fit_transform(X_train)

# Aplicar a mesma transformação nos dados de teste
X_test_codificado = preprocessador.transform(X_test)

### Balanceamento das classes

Esta etapa busca lidar com o desbalanceamento entre as classes de sinistros fatais e não fatais. Como os acidentes fatais são menos frequentes, serão avaliadas técnicas para reduzir o impacto dessa diferença e melhorar a capacidade do modelo de identificar casos fatais.

## Modelos de classificação

### Logistic Regression (Regressão Logística)
A **Logistic Regression** é um algoritmo de aprendizado supervisionado usado principalmente para classificação, especialmente em problemas de classificação binária. Neste projeto, será utilizada para classificar os sinistros de trânsito como **fatais** ou **não fatais**.

In [4]:
# Config do modelo
model_lr = LogisticRegression( 
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)

# Treinar modelo com os dados de treino
model.fit(X_train, y_train)

# 
y_pred = model.predict(x_test)

### Matriz de Confusão

A matriz de confusão será utilizada para avaliar o desempenho dos modelos de classificação, comparando as classes reais com as previstas. Essa análise permitirá identificar quantos sinistros fatais e não fatais foram classificados corretamente ou incorretamente, destacando possíveis dificuldades na identificação de acidentes fatais.

### Random Forest (Floresta Aleatória)
O **Random Forest** é um algoritmo de aprendizado supervisionado usado principalmente para classificação, ele combina várias árvores de decisão para realizar uma classificação. Essa abordagem pode identificar relações complexas entre as métricas dos acidentes, contribuindo para analisar quais fatores estão associados à ocorrência de sinistros fatais.

In [6]:
model_rf = RandomForestClassifier()




#### Matriz de Confusão

### Decision Tree (Árvore de Decisão)

A **Decision Tree** é um algoritmo e aprendizado supervisionado usado tanto para classificação quanto para regressão, utiliza regras e condições para classificar os dados. Neste projeto, poderá identificar padrões entre as métricas dos acidentes e a gravidade dos sinistros. Uma de suas principais vantagens é a facilidade de interpretação dos critérios utilizados pelo modelo.

In [ ]:
model_dtc = DecisionTreeClassifier()




#### Matriz de Confusão

### Validação cruzada (Cross-Validation)

A validação cruzada consiste em dividir os dados de treinamento em diferentes partes (folds) para treinar e avaliar o modelo em combinações distintas. Essa técnica permite analisar a estabilidade do desempenho e reduz a dependência de uma única divisão dos dados.

Com dados desbalanceados, um único split treino/teste pode ser "sortudo" ou "azarado". Por exemplo:

Se por acaso os fatais ficaram mais concentrados no treino → modelo aprende melhor.                                                                   
Se ficaram mais no teste → métricas ruins

O **k-fold cross-validation** repetiria o processo 5 ou 10 vezes com divisões diferentes e mostraria a variação real da performance. Sem isso, as métricas apresentadas são uma foto de um único momento, potencialmente não representativa.

## Comparação dos modelos

Nesta etapa, os resultados da Regressão Logística, Random Forest e Árvore de Decisão serão comparados utilizando métricas de avaliação, como precision, recall, F1-score e matriz de confusão. O objetivo é identificar qual modelo apresenta o desempenho mais adequado para classificar os sinistros como fatais ou não fatais.

- Accuracy → quantas previsões acertou no total;
- Precision → quando previu determinada classe, quantas vezes estava correto;
- Recall → quantos casos daquela classe conseguiu identificar;
- F1-Score → equilíbrio entre Precision e Recall.
- Matriz de Confusão → mostra detalhadamente os acertos e erros por classe.

In [ ]:
report_random_forest = classification_report()
report_logistic_regression = classification_report()
report_decision_tree = classification_report()

# Criar tabela dinâmica para comparação.
resultados = pd.DataFrame({
    "Modelo": [
        "Random Forest",
        "Logistic Regression",
        "Decision Tree"
    ],
    "Tipo": [
        "Supervisionado",
        "Supervisionado",
        "Supervisionado"
    ],
    "Precision": [
        report_random_forest["1"]["precision"],
        report_logistic_regression["1"]["precision"],
        report_decision_tree["1"]["precision"]
    ],
    "Recall": [
        report_random_forest["1"]["recall"],
        report_logistic_regression["1"]["recall"],
        report_decision_tree["1"]["recall"]
    ],
    "F1-Score": [
        report_random_forest["1"]["f1-score"],
        report_logistic_regression["1"]["f1-score"],
        report_decision_tree["1"]["f1-score"]
    ]
})

# Mostrar tabela. 
resultados.round(2) # "round" limita casas decimais.

## análise de erros (opcional)

Nesta etapa, serão investigadas as previsões incorretas dos modelos para identificar padrões e possíveis limitações na classificação dos sinistros. O objetivo será compreender em quais situações o modelo apresenta maior dificuldade, especialmente na identificação de acidentes fatais, contribuindo para interpretar os resultados e identificar oportunidades de melhoria.

## Conclusão